# Project 2.1 — Monte Carlo Radioactive Decay
**Topic:** Simulating nuclear decay as a Bernoulli process; connecting stochasticity to the deterministic exponential law.


---
## 1. Set Up

### Physical Motivation
Radioactive decay is the archetypal **quantum stochastic process**: each nucleus independently decays in any time interval $dt$ with probability $\lambda\,dt$, where $\lambda = \ln 2 / t_{1/2}$ is the decay constant. In the $N \to \infty$ limit, the ensemble average obeys the exact deterministic law:

$$\langle N(t) \rangle = N_0 e^{-\lambda t}$$

But for **finite $N$**, fluctuations scale as $\sqrt{N}$ and are physically observable. This project bridges the quantum-stochastic picture and the bulk exponential law.

### Why Binomial Thinning, Not $p = \lambda\,\Delta t$?
The naive approximation $p = \lambda\,\Delta t$ is only valid for $\lambda\,\Delta t \ll 1$. The exact probability that a nucleus survives a time step $\Delta t$ is $e^{-\lambda\Delta t}$, so the exact decay probability per step is:

$$p = 1 - e^{-\lambda\Delta t}$$

Using $p \approx \lambda\Delta t$ causes systematic overestimation of the decay rate. The code uses the exact formula — this is a crucial subtlety.

### Method — Binomial Thinning
At each time step, the number of nuclei that decay out of $N_{\text{current}}$ surviving nuclei follows a **Binomial distribution**:

$$\Delta N \sim \text{Binomial}(N_{\text{current}},\, p)$$

This is exact: each nucleus independently decays with probability $p$. The implementation vectorises over `n_trials` independent experiments simultaneously.

### Parameters
| Symbol | Meaning | Typical value |
|--------|---------|---------------|
| $N_0$ | Initial nuclei | $10^4$ |
| $t_{1/2}$ | Half-life | $10\,\text{s}$ |
| $\Delta t$ | Time step | $0.5\,\text{s}$ |
| $n_{\text{trials}}$ | Number of MC realisations | $200$ |

### ⚠️ Limitations
1. **Independence assumption**: assumes no daughter product interference, no induced fission — valid for simple decay chains only.
2. **Integer nuclei**: the code uses `int64` — correct for real nuclei. The continuous exponential is an approximation.
3. **Finite $n_{\text{trials}}$**: the ensemble mean converges to the analytic curve as $n_{\text{trials}}^{-1/2}$. With 200 trials, relative error in the mean is $\sim 7\%$.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_decay_mc")
OUT.mkdir(exist_ok=True)

# Fixed seed for reproducibility. Change seed to verify results are seed-independent.
rng = np.random.default_rng(20260526)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `simulate_decay_binomial()`

```python
lam = np.log(2.0) / half_life
```
Converts half-life to decay constant. **Derivation**: $N(t_{1/2}) = N_0/2 = N_0 e^{-\lambda t_{1/2}}$ implies $\lambda = \ln 2/t_{1/2}$.

```python
p = 1.0 - np.exp(-lam * dt)
```
Exact per-step decay probability. For small $\lambda\Delta t$: $p \approx \lambda\Delta t - \frac{(\lambda\Delta t)^2}{2} + \ldots$. The $\mathcal{O}(\lambda^2\Delta t^2)$ correction matters when $\Delta t \sim t_{1/2}$.

```python
counts = np.empty((n_trials, t.size), dtype=float)
```
Pre-allocates the output array: `n_trials` rows (independent experiments), `t.size` columns (time points). Pre-allocation avoids repeated memory reallocation in the time loop.

```python
decayed = rng.binomial(current, p)
current = current - decayed
```
**Core MC step.** `rng.binomial(n, p)` draws $n_{\text{trials}}$ independent samples simultaneously — vectorised over all trials. This is $N(t+\Delta t) = N(t) - \text{Binomial}(N(t), p)$.

**Why not `rng.binomial(N0, p)` directly (without tracking survivors)?** Because after the first step, the population is $N_0 - \Delta N_1$, which differs across trials. You must track each trial's current population independently.

### `fit_decay_constant()`

```python
coeff = np.polyfit(t[mask], np.log(mean_counts[mask]), deg=1)
```
Linear fit to $\log N$ vs. $t$. Since $\log N(t) = \log N_0 - \lambda t$, the slope gives $-\hat\lambda$. **Warning**: `np.polyfit` minimises $\sum (\log N - \text{model})^2$, which gives more weight to large-$t$ points where $N$ is small and statistical noise is large. A weighted fit using $w_i = N_i$ (inverse variance weights) is more appropriate.

### 🔧 Improvement Directions
1. **Weighted least-squares fit**: use `np.polyfit(t, log_N, 1, w=mean_counts)` to weight early-time points more heavily.
2. **Variance analysis**: check that $\text{Var}[N(t)] \approx N_0 e^{-\lambda t}(1 - e^{-\lambda t})$ — the exact binomial variance. Plot this as a shaded band.
3. **Two-component decay**: add a daughter product $B$ that accumulates. This gives $N_A(t) = N_0 e^{-\lambda_A t}$, $N_B(t) = N_0\frac{\lambda_A}{\lambda_B - \lambda_A}(e^{-\lambda_A t} - e^{-\lambda_B t})$.
4. **Poisson limit**: for $N \gg 1$ and small $p$, `Binomial(N,p) ≈ Poisson(Np)`. Verify this numerically.

### ⚠️ Known Weaknesses
- **Large-N regime only**: when $N \to 0$, discrete fluctuations dominate and the exponential fit becomes unreliable.
- **Seed dependence**: with a fixed seed, results are reproducible but not truly random. Always verify conclusions across multiple seeds.


In [ ]:
def simulate_decay_binomial(N0, half_life, t_max, dt, n_trials=200, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    lam = np.log(2.0) / half_life          # decay constant lambda = ln2/t_half
    p   = 1.0 - np.exp(-lam * dt)          # EXACT per-step decay probability
    t   = np.arange(0.0, t_max + 0.5*dt, dt)

    counts = np.empty((n_trials, t.size), dtype=float)
    counts[:, 0] = N0                       # all trials start at N0

    current = np.full(n_trials, N0, dtype=np.int64)
    for j in range(1, t.size):
        decayed   = rng.binomial(current, p)     # vectorised over all trials
        current   = current - decayed
        counts[:, j] = current

    mean  = counts.mean(axis=0)
    std   = counts.std(axis=0, ddof=1)
    exact = N0 * np.exp(-lam * t)               # deterministic exponential law
    return t, mean, std, exact, counts

def fit_decay_constant(t, mean_counts, min_count=1.0):
    mask  = mean_counts > min_count             # exclude near-zero tail (noisy log)
    coeff = np.polyfit(t[mask], np.log(mean_counts[mask]), deg=1)
    lam_fit = -coeff[0]                         # slope of log(N) vs t is -lambda
    return lam_fit, coeff[1]

def summarize_isotope(label, N0, half_life, t_max, dt, n_trials, rng):
    t, mean, std, exact, counts = simulate_decay_binomial(
        N0=N0, half_life=half_life, t_max=t_max, dt=dt,
        n_trials=n_trials, rng=rng)
    lam_fit, _ = fit_decay_constant(t, mean)
    lam_true   = np.log(2.0) / half_life
    return dict(label=label, N0=N0, half_life=half_life,
                t=t, mean=mean, std=std, exact=exact, counts=counts,
                lambda_true=lam_true, lambda_fit=lam_fit,
                half_life_fit=np.log(2.0)/lam_fit)


---
## 3 & 4. Calculation & Writeouts

### What to Expect
- The ensemble mean $\bar N(t)$ should closely follow $N_0 e^{-\lambda t}$ — deviations are $\mathcal{O}(1/\sqrt{n_{\text{trials}}})$.
- The standard deviation across trials should scale as $\sqrt{N(t)}$ — binomial fluctuations shrink relative to the mean as $N\to\infty$.
- The fitted $\hat\lambda$ should agree with $\lambda_{\text{true}}$ to within a few percent.

### Physical Insight
The relative fluctuation $\sigma_N/\langle N\rangle = 1/\sqrt{N(t)}$ grows over time as $N(t)$ decreases. This is why **counting statistics** matter more for rare isotopes or late decay times — the signal-to-noise is fundamentally limited by Poisson statistics.


In [ ]:
isotopes = [
    dict(label="Isotope A",  N0=10000, half_life=10.0, t_max=60.0,  dt=0.5),
    dict(label="Isotope B",  N0=5000,  half_life=25.0, t_max=120.0, dt=1.0),
]

results = []
for iso in isotopes:
    r = summarize_isotope(**iso, n_trials=200, rng=rng)
    results.append(r)
    print(f"{r['label']}:")
    print(f"  lambda_true = {r['lambda_true']:.6f} s⁻¹")
    print(f"  lambda_fit  = {r['lambda_fit']:.6f} s⁻¹")
    print(f"  half_life_fit = {r['half_life_fit']:.4f} s  (true = {r['half_life']:.1f} s)")
    print(f"  relative error: {abs(r['lambda_fit']-r['lambda_true'])/r['lambda_true']*100:.2f}%")
    print()


---
## 5. Matplotlib Visualisation

### Interpreting the Plot
- **Solid curves**: ensemble mean $\bar N(t)$ — one per isotope.
- **Shaded band**: $\bar N(t) \pm \sigma(t)$ — the $1\sigma$ spread across trials. This is NOT the uncertainty in the mean; it is the physical fluctuation between individual decay experiments.
- **Dashed**: analytic exponential $N_0 e^{-\lambda t}$.
- **Semilog y-axis**: transforms the exponential into a straight line. Deviations from linearity at late times are where discrete fluctuations dominate.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, logy in zip(axes, [False, True]):
    for r, c in zip(results, ['steelblue', 'tomato']):
        t, mean, std, exact = r['t'], r['mean'], r['std'], r['exact']
        ax.plot(t, mean, color=c, lw=2, label=f"{r['label']} MC mean")
        ax.fill_between(t, np.maximum(mean-std,0), mean+std, alpha=0.2, color=c)
        ax.plot(t, exact, '--', color=c, lw=1.2, alpha=0.7, label=f"Analytic $e^{{-\lambda t}}$")
    if logy:
        ax.set_yscale('log')
        ax.set_title('Semilog — linearity confirms exponential', fontsize=11)
    else:
        ax.set_title('Linear scale — fluctuation bands', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('$N(t)$')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "decay_mc.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Sanity Check & Validation

| Test | Formula | Tolerance |
|------|---------|-----------|
| $\bar N(0) = N_0$ | Mean at $t=0$ | $< 0.1$ |
| Variance ~ Binomial | $\text{Var}[N(t)] \approx N_0 p_t(1-p_t)$ where $p_t = e^{-\lambda t}$ | within $10\%$ |
| Fitted $\hat\lambda$ | $|\hat\lambda - \lambda| / \lambda < 0.05$ | $5\%$ relative |
| Monotone decrease | $\bar N(t+\Delta t) \le \bar N(t)$ | strict |


In [ ]:
print("=" * 55)
print("SANITY CHECKS — Project 02.1 Radioactive Decay MC")
print("=" * 55)

for r in results:
    t, mean, std, exact = r['t'], r['mean'], r['std'], r['exact']
    N0, lam = r['N0'], r['lambda_true']
    print(f"\n── {r['label']} ──")

    # 1. Initial condition
    ok = abs(mean[0] - N0) < 1
    print(f"  1. N(0) = {mean[0]:.1f}  (expected {N0})  {'✓' if ok else '✗'}")

    # 2. Lambda fit accuracy
    rel_err = abs(r['lambda_fit'] - lam) / lam
    ok = rel_err < 0.05
    print(f"  2. λ_fit relative error = {rel_err*100:.2f}%  {'✓' if ok else '✗'}")

    # 3. Monotone decrease of mean
    monotone = np.all(np.diff(mean) <= 0.5)  # allow tiny uptick from rounding
    print(f"  3. Monotone decrease: {'✓' if monotone else '✗'}")

    # 4. Variance check at t = t_half
    idx_half = np.argmin(np.abs(t - r['half_life']))
    pt = np.exp(-lam * t[idx_half])
    var_expected = N0 * pt * (1 - pt)
    var_actual   = std[idx_half]**2
    ratio = var_actual / var_expected if var_expected > 0 else np.inf
    ok = 0.5 < ratio < 2.0
    print(f"  4. Var ratio at t_half: {ratio:.3f}  (expected ≈1.0)  {'✓' if ok else '✗'}")

print("\n" + "=" * 55)
